# Import Packages

In [1]:
from glob import glob
import random
import warnings
import matplotlib.pyplot as plt
from cmcrameri import cm
import seaborn as sns
import numpy as np
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
import contextily as ctx
import geopandas as gpd
#from numpy import abs, nanpercentile, arange
from pathlib import Path
from matplotlib.cm import ScalarMappable

import func_gev as gev
import func_preparation as dbf
import func_plotting as dbplt
import func_utils as ut

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_style('whitegrid')
%matplotlib inline
CMAP = cm.managua_r

fs = 12

# Settings

In [2]:
path_data = '../input/Annual_max_DCPP_20260112/'
path_export = '../output/exploration/'

dir_results = '../output/gev_analysis/2026-03-20/'
dir_data_pkl = '../output/gev_analysis/2026-03-20/data.parquet'

In [3]:
HINDCAST_START = 1960
HINDCAST_END = 2026

In [4]:
axes_color: str = '#333333'
markers_color: str = "#99E3DDFF"
colors_reg: list = ['#333333FF', '#7F6C7BFF']  

In [5]:
RETURN_PERIODS = [10, 25, 50, 100, 200]
PLOT_PERIOD_EVOLUTION = ['10-year', '50-year', '100-year']

In [6]:
CONFIDENCE_INTERVAL = 0.9 # two-side 90% CI interval (5% each side)

In [7]:
# adding a little randomness to enhance trust the model works as robust as possible cross locations..
start_location = 0 #random.choice(arange(0, 9579))
end_location = 2000 #start_location+10

print(f'analyse a subsample of location {start_location}–{end_location}')

analyse a subsample of location 0–2000


# Import Data

### Import Raw Data from Models

In [ ]:
ls_files = [file for file in glob(path_data + '*.nc')]
ls_files

In [ ]:
dic_data_per_model = dbf.import_all_models(ls_files)

### Import regression results for location trend regression

In [8]:
[
    results_annual_stat_all, results_nonstat_all, location_geo_info, location_point_info
    ] = ut.import_info_for_regression(dir_results)

loading ../output/gev_analysis/2026-03-20//stationary_per_year.pkl
loading ../output/gev_analysis/2026-03-20//nonstationary.pkl
loading ../output/gev_analysis/2026-03-20//LatLon.pkl
loading ../output/gev_analysis/2026-03-20//location_info.pkl


In [10]:
results_stat_all = ut.import_pickle_data(dir_results, 'stationary.pkl')

loading ../output/gev_analysis/2026-03-20//stationary.pkl


# Data Processing...

### BiasCorrection,...

In [ ]:
dic_data_per_model, combined, notes_overview = dbf.prepare_combined_data(ls_files, dic_data_per_model)

In [ ]:
dic_data_per_location = dbf.extract_location_data(combined, HINDCAST_START, HINDCAST_END)

# VISUALS

## Map of Missing Data

In [ ]:
n_obs_per_location = dbf.get_number_of_observations_per_site(dic_data_per_location)
n_obs_per_location

In [ ]:
df_missing_location, df_valid, fig_map = dbf.create_summary_location_w_missing_data(
    dic_data_per_model=dic_data_per_model, combined=combined, n_obs_per_location=n_obs_per_location, dir_export='output/exploration/'
    )

In [ ]:
display(fig_map)

## μ Regression - Non-stationary vs annual stationary per location

compute with standardized or non-standardized years

##### Compute for 1 Location

In [ ]:
ls_missing = []
loc_id = random.choice(list(results_annual_stat_all.keys()))

print(f'Compute regression analysis for μ₁ for site {loc_id}...')

In [ ]:
if results_nonstat_all[loc_id] is None:
    ls_missing.append(loc_id)
    print(f'Could not process side {loc_id}, NO nonstationary results available (counting {len(ls_missing)})')

years_, dic_trend = gev.prepare_for_regression(    
    annual_stationary=results_annual_stat_all[loc_id], 
    nonstationary=results_nonstat_all[loc_id],
    confidence_level_pc=CONFIDENCE_INTERVAL, factor_m_to_mm=1000,
    years_scaled=True
    ) 

In [ ]:
fig_reg = dbplt.plot_location_regression(
    loc_id, years_, dic_trend, results_annual_stat_all[loc_id], 
    confidence_level_pc=CONFIDENCE_INTERVAL, display_results=True, years_scaled=True,
    axes_color='#333333', markers_color="#99E3DDFF", colors_reg=['#CAA5C2FF', '#005C55FF'], 
    fontsize=12, 
    )

#### Compute for All

In [ ]:
dic_fig_regression = {}
ls_missing = []
for loc_id in results_annual_stat_all.keys():
    if loc_id >= start_location and loc_id <= end_location:
        print(f'processing site {loc_id}...')
        
        if results_nonstat_all[loc_id] is None:
            ls_missing.append(loc_id)
            print(f'Skipping side {loc_id}, no nonstationary results (counting {len(ls_missing)})')
            dic_fig_regression[loc_id] = None
            continue
        
        years_, dic_trend = gev.prepare_for_regression(            
            annual_stationary=results_annual_stat_all[loc_id], nonstationary=results_nonstat_all[loc_id],
            confidence_level_pc=CONFIDENCE_INTERVAL, factor_m_to_mm=1000, years_scaled=True
            ) 

        fig_reg = dbplt.plot_location_regression(
            loc_id, years_, dic_trend, results_annual_stat_all[loc_id], fontsize=12,
            confidence_level_pc=CONFIDENCE_INTERVAL, years_scaled=True,
            axes_color='#333333', markers_color="#99E3DDFF", colors_reg=['#CAA5C2FF',  '#005C55FF'], 
            )
        
        dic_fig_regression[loc_id] = fig_reg

In [ ]:
for loc_id in dic_fig_regression.keys():
    if dic_fig_regression[loc_id]:
        ut.store_location_regression(
            dic_fig_regression[loc_id],loc_id, 
            location_geo_info[loc_id], location_point_info[loc_id], 
            Path('../output/gev_analysis/2026-03-20/')
            )

## Maps of Fit Parameters

In [11]:
OUTLIER_MARKER_SIZE = 6

def remove_nan_sites(df_plot):
    ls_nan_loc = []
    for en, a in enumerate(df_plot.alpha):
        if np.isnan(a):
            ls_nan_loc.append(en)
    print(f'{len(ls_nan_loc)} missing location information')

    return df_plot.dropna()


def initialize_projection(df_clean, parameter, min_size, max_size, mark_outlier):
    gdf = gpd.GeoDataFrame(
        df_clean, geometry=gpd.points_from_xy(df_clean['lon'], df_clean['lat']),
        crs="EPSG:4326"
    )
    gdf_web = gdf.to_crs(epsg=3857)
    gdf_web = dbplt.custom_color_for_mu_and_mu1(
        gdf_web,  parameter=parameter, mark_outlier=mark_outlier, min_size=min_size, max_size=max_size
        )

    colors = []
    for c in gdf_web['color']:
        c_rgba = list(c)
        if len(c_rgba) == 3:
            c_rgba.append(255)
        colors.append([v/255 for v in c_rgba])
    colors = np.array(colors)

    if mark_outlier is True:
        gdf_web["marker_size"] = gdf_web.apply(
            lambda row: OUTLIER_MARKER_SIZE if row["outliers"] else row["marker_size"],
            axis=1
        )
        
        valid = gdf_web.loc[~gdf_web["outliers"], parameter]
        min_value = np.nanmin(valid)
        max_value = np.nanmax(valid)

    else:
        max_value = np.nanmax(gdf_web[parameter])
        min_value = np.nanmin(gdf_web[parameter])  
        
    norm = mcolors.Normalize(vmin=min_value, vmax=max_value)
    return gdf_web, colors, norm


def plot_mu_and_mu1_for_all_sites_static(
    df_clean, approach, parameter, min_size, max_size, mark_outlier, unit,
    edgecolor_marker=None, lw_marker=0.1, figsize=(8,5), fontsize=10, display_results:bool=True, save_plot:bool=False, 
    save_path:str|None=None, file_name:str|None=None, threshold_z_method:float|None=None
    ):
    if parameter == 'mu1':
        title = f"Map of location parameter $μ_1$ · {approach}"
        label_colormap = f"Location trend $μ_1$, {unit}"
    elif parameter == 'mu':
        title = f"Map of location parameter $μ$ · {approach}"
        label_colormap = f"Location trend $μ$, {unit}"
        
    gdf_web, colors, norm = initialize_projection(df_clean, parameter, min_size, max_size, mark_outlier)

    fig, ax = plt.subplots(figsize=figsize)

    sc = ax.scatter(
            gdf_web.geometry.x, gdf_web.geometry.y,
            c=colors,
            s=gdf_web['marker_size'],
            edgecolor=edgecolor_marker, linewidth=lw_marker 
    )

    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, zoom=6)
    if mark_outlier:
        legend_elements = [
            Line2D([0], [0], linestyle='None', marker=None, label=f"Marker size ~ uncertainty (high → large)\nOutlier marked using modified Z-score (threshold {threshold_z_method:.2f}·σ)")
        ]
        ax.legend(handles=legend_elements, loc='lower right', fontsize=fontsize*0.8, framealpha=0.6)
    else:
        legend_elements = [
            Line2D([0], [0], linestyle='None', marker=None, label=f"Marker size ~ uncertainty (high → large)")
        ]
        ax.legend(handles=legend_elements, loc='lower right', fontsize=fontsize*0.8, framealpha=0.6)
        
    ax.set_axis_off()
    sm = ScalarMappable(cmap=CMAP, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, orientation='vertical', fraction=0.03, pad=0.04)
    cbar.set_label(label_colormap, fontsize=fontsize)

    plt.title(title, loc='left', fontsize=fontsize)

    plt.tight_layout()
    if save_plot:
        if save_path is None:
            raise ValueError('Missing path for output; provide `save_path`...')
        if file_name is None:
            raise ValueError('Missing file_name for output; provide `file_name`...')

        save_dir = Path(save_path)
        save_dir.mkdir(parents=True, exist_ok=True) 
    
        file_name = (save_dir / file_name).with_suffix('.png')
        plt.savefig(file_name, dpi=300, bbox_inches='tight')
    
    plt.show() if display_results else plt.close(fig)
    return fig

In [12]:
def plot_location_map(
    approach, parameter, unit, df, location_geo_info, threshold_z_outlier, mark_outlier, interactive_map, cmap,
    min_size, max_size, inital_zoom=3.5, saving_map:bool=False, save_path:str|None=None, fs=12
    ):
    if parameter not in ('mu', 'mu0', 'mu1'):
        raise ValueError('Wrong parameter selected! Make sure you want to plot location or location trend...')
    if parameter == 'mu0':
        parameter = 'mu'
        
    # ------ preparation ------
    df_plot, df_mu = dbplt.prep_for_plot(
        result=df, location_geo_info=location_geo_info, approach=approach, parameter=parameter,
        confidence_level_pc=CONFIDENCE_INTERVAL, threshold_z_outlier=threshold_z_outlier, 
        mark_outlier=mark_outlier
        )
    print(f'\nnon-stationary data: {df_plot.shape} out of {df_mu.shape}')
    
    if parameter == 'mu' and unit == 'm':
        df_plot['mu'] = df_plot.mu/1000
        
    df_plot = dbplt.custom_color_for_mu_and_mu1(
        df_plot, parameter=parameter, mark_outlier=mark_outlier, min_size=min_size, max_size=max_size
        )
    
    print(f'Parameter Overview {parameter}, {unit}')
    print(df_plot[parameter].describe())
    print()
    
    # ------ plotting ------ 
    if interactive_map is True:
        if mark_outlier is True:
            file_name = "Map_μ₁_inclCI_markedOutlier" if parameter == 'mu1' else "Map_μ_inclCI_markedOutlier"
        else:
            file_name = "Map_μ₁_inclCI_all" if parameter == 'mu1' else "Map_μ_inclCI_all"
                
        fig_map = dbplt.plot_mu_and_mu1_for_all_sites_interactive(
            df_plot_=df_plot, approach=approach, parameter=parameter, unit=unit, mark_outlier=mark_outlier, 
            threshold_z_method=threshold_z_outlier, inital_zoom=inital_zoom, saving_map=saving_map, cmap=cmap, 
            file_name=file_name, 
            )
    else:
        if mark_outlier is True:
            file_name = f'Map_μ₁_inclCI_markedOutlier_{approach}' if parameter == 'mu1' else f'Map_μ_inclCI_markedOutlier_{approach}'
        else:
            file_name = f'Map_μ₁_inclCI_all_{approach}' if parameter == 'mu1' else f'Map_μ_inclCI_all_{approach}'

        df_clean = remove_nan_sites(df_plot)
        fig_map = plot_mu_and_mu1_for_all_sites_static(
            df_clean, parameter=parameter, unit=unit, approach=approach, min_size=min_size, 
            max_size=max_size, mark_outlier=mark_outlier, threshold_z_method=threshold_z_outlier, 
            display_results=True, save_plot=True, fontsize=fs, file_name=file_name, save_path=save_path, 
            )
    return fig_map

### scale parameter

#### interactive PyDeck

#### static Matplotlib

### shape parameter

#### interactive PyDeck

#### static Matplotlib

### location parameter

In [ ]:
parameter='mu'
unit = 'm'

mark_outlier = False

approach = 'stationary' #'annual-stationary', 'non-stationary', 'stationary'
threshold_z_outlier = 3

interactive_map = True
min_size = 2000 if interactive_map is True else 2.5
max_size = 20000 if interactive_map is True else 35

export_path = '../output/gev_analysis/2026-03-24/figures/'

In [ ]:
if approach == 'non-stationary':
    df=results_nonstat_all
elif approach == 'annual-stationary':
    df=results_annual_stat_all
elif approach == 'stationary':
    df=results_stat_all

fig_map = plot_location_map(
    approach=approach, parameter=parameter, unit=unit, df=df, location_geo_info=location_geo_info, 
    threshold_z_outlier=threshold_z_outlier, mark_outlier=mark_outlier, interactive_map=interactive_map, 
    min_size=min_size, max_size=max_size, cmap=CMAP, saving_map=True, save_path=export_path, fs=12
    )

#### Stationary Approach

In [ ]:
threshold_z_outlier_stat = 3

### μ₁ Trend

$μ_1$ with the color indicating the absolute value and the size indicating the std of $μ_1$ 

In [ ]:
parameter='mu1'
unit = 'mm/year'

mark_outlier = False

approach = 'annual-stationary' #'annual-stationary' #'non-stationary'
threshold_z_outlier = 3

interactive_map = True
min_size = 2000 if interactive_map is True else 2.5
max_size = 20000 if interactive_map is True else 35

export_path = '../output/gev_analysis/2026-03-24/figures/'

In [ ]:
if approach == 'non-stationary':
    df=results_nonstat_all
elif approach == 'annual-stationary':
    df=results_annual_stat_all

fig_map = plot_location_map(
    approach=approach, parameter=parameter, unit=unit, df=df, location_geo_info=location_geo_info, 
    threshold_z_outlier=threshold_z_outlier, mark_outlier=mark_outlier, interactive_map=interactive_map, 
    min_size=min_size, max_size=max_size, cmap=CMAP, saving_map=True, save_path=export_path, fs=12
    )